# Chapter 8: Text Summarization

Text summarization is a valuable NLP technique that generates concise, coherent summaries from larger bodies of text. The goal is to retain the most important information while significantly reducing the amount of text.

**Two main approaches:**
- **Extractive Summarization** -- Selects and extracts key sentences directly from the original text
- **Abstractive Summarization** -- Generates new sentences that convey the same meaning as the original

In this notebook, we will implement both approaches, explore advanced techniques, and learn how to evaluate summarization quality.

## Setup and Installation

Install the required packages before running the notebook.

In [ ]:
# Install required packages
!pip install nltk numpy networkx scikit-learn transformers torch rouge-score

In [ ]:
# Download NLTK resources
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

---
## 8.1 Extractive Summarization

Extractive summarization selects the most important sentences from the original text and combines them to form a summary. It relies on identifying key sentences based on criteria such as:

- **Term Frequency** -- How often important words appear in a sentence
- **Sentence Position** -- Where the sentence appears in the text (first/last sentences are often important)
- **Similarity to Title** -- How closely the sentence aligns with the main topic

This approach is straightforward and computationally efficient, though it may not always produce the most coherent summaries since sentences are selected independently.

### 8.1.1 Understanding Extractive Summarization

The main steps involved in extractive summarization are:

1. **Preprocessing** -- Tokenization, stop word removal, and normalization
2. **Sentence Scoring** -- Each sentence is assigned a score based on features like term frequency
3. **Sentence Selection** -- The highest-scoring sentences are selected for the summary
4. **Summary Generation** -- Selected sentences are combined in logical order

### 8.1.2 Implementing Extractive Summarization with NLTK

We will use the `nltk` library to implement a simple extractive summarization system based on term frequency scoring.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.cluster.util import cosine_distance
import numpy as np
import networkx as nx

# Sample text
text = """Natural language processing (NLP) is a subfield of linguistics, computer
science, and artificial intelligence
concerned with the interactions between computers and human language, in particular
how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language
processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# Preprocess the text
sentences = sent_tokenize(text)
stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence):
    """Tokenize, lowercase, and remove stopwords and non-alphanumeric tokens."""
    words = word_tokenize(sentence.lower())
    words = [word for word in words if word.isalnum() and word not in stop_words]
    return words

# Sentence scoring based on term frequency
def score_sentences(sentences):
    """Score each sentence by summing the frequencies of its words."""
    sentence_scores = []
    word_frequencies = FreqDist(
        [word for sentence in sentences for word in preprocess_sentence(sentence)]
    )
    for sentence in sentences:
        words = preprocess_sentence(sentence)
        sentence_score = sum(word_frequencies[word] for word in words)
        sentence_scores.append((sentence, sentence_score))
    return sentence_scores

# Select top-ranked sentences
def select_sentences(sentence_scores, num_sentences=2):
    """Sort sentences by score and return the top ones."""
    sentence_scores.sort(key=lambda x: x[1], reverse=True)
    selected_sentences = [sentence[0] for sentence in sentence_scores[:num_sentences]]
    return selected_sentences

# Generate summary
sentence_scores = score_sentences(sentences)
summary_sentences = select_sentences(sentence_scores)
summary = ' '.join(summary_sentences)

print("Original Text:")
print(text)
print("\n" + "="*80)
print("\nSentence Scores:")
for sent, score in score_sentences(sentences):
    print(f"  Score {score:3d}: {sent[:80]}..." if len(sent) > 80 else f"  Score {score:3d}: {sent}")
print("\nSummary:")
print(summary)

**How it works:**

1. The text is split into sentences using `sent_tokenize`
2. Each sentence is preprocessed by tokenizing, lowercasing, and removing stopwords
3. A word frequency distribution is computed across all sentences
4. Each sentence is scored by summing the frequencies of its constituent words
5. The top-scoring sentences are selected to form the summary

This is a simple but effective baseline approach. Sentences with more frequent (and presumably important) terms receive higher scores.

### 8.1.3 Advanced Extractive Summarization: TextRank

**TextRank** is a graph-based ranking algorithm adapted from Google's PageRank. In summarization:

- Each sentence is a **node** in the graph
- **Edges** between nodes represent the similarity between sentences
- Sentences that are similar to many other sentences receive higher ranks

This captures the idea that important sentences are those that are central to the overall meaning of the text.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.cluster.util import cosine_distance
import numpy as np
import networkx as nx

# Sample text
text = """Natural language processing (NLP) is a subfield of linguistics, computer
science, and artificial intelligence
concerned with the interactions between computers and human language, in particular
how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language
processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# Preprocess the text
sentences = sent_tokenize(text)
stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence):
    words = word_tokenize(sentence.lower())
    words = [word for word in words if word.isalnum() and word not in stop_words]
    return words

# Build sentence similarity matrix using cosine distance
def build_similarity_matrix(sentences):
    """Create a matrix where each cell [i][j] represents the similarity between sentence i and j."""
    similarity_matrix = np.zeros((len(sentences), len(sentences)))
    for i, sentence1 in enumerate(sentences):
        for j, sentence2 in enumerate(sentences):
            if i != j:
                words1 = preprocess_sentence(sentence1)
                words2 = preprocess_sentence(sentence2)
                similarity_matrix[i][j] = 1 - cosine_distance(words1, words2)
    return similarity_matrix

# Apply TextRank algorithm
def textrank(sentences, num_sentences=2):
    """Rank sentences using the PageRank algorithm on the similarity graph."""
    similarity_matrix = build_similarity_matrix(sentences)
    similarity_graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(similarity_graph)
    ranked_sentences = sorted(
        ((scores[i], sentence) for i, sentence in enumerate(sentences)),
        reverse=True
    )
    selected_sentences = [sentence for score, sentence in ranked_sentences[:num_sentences]]
    return selected_sentences

# Generate summary
summary_sentences = textrank(sentences)
summary = ' '.join(summary_sentences)

print("TextRank Summary:")
print(summary)

**Key steps in TextRank:**

1. **Preprocessing** -- Tokenize into sentences and words, remove stopwords
2. **Similarity Matrix** -- Compute cosine similarity between all pairs of sentences
3. **Graph Construction** -- Build a graph from the similarity matrix using NetworkX
4. **PageRank** -- Apply the PageRank algorithm to score each sentence
5. **Selection** -- Pick the top-ranked sentences for the summary

### 8.1.4 Advanced Technique: Latent Semantic Analysis (LSA)

**LSA** is an unsupervised learning technique that captures the latent (hidden) structure of text using **Singular Value Decomposition (SVD)**. It decomposes a term-document matrix to identify patterns in the relationships between terms and documents, enabling a more nuanced understanding of text content.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

# Sample documents
documents = [
    "The cat sat on the mat",
    "The dog chased the ball",
    "The bird flew in the sky"
]

# Create a term-document matrix
vectorizer = CountVectorizer()
term_document_matrix = vectorizer.fit_transform(documents)

# Perform Singular Value Decomposition (SVD)
svd = TruncatedSVD(n_components=2)  # Reduce dimensionality to 2
lsa_matrix = svd.fit_transform(term_document_matrix)

# Reduced term and document representations (topics)
terms = vectorizer.get_feature_names_out()
lsa_topics = svd.components_

# Print the results
print("Terms:", terms)
print("\nReduced Document Representations (Topics):")
print(lsa_matrix)
print("\nReduced Term Representations (Topics):")
print(lsa_topics)

**How LSA works for summarization:**

- The term-document matrix captures word occurrences across documents
- SVD decomposes this matrix into lower-dimensional representations ("topics")
- Sentences that align strongly with the dominant topics are considered most important
- This goes beyond simple word frequency by capturing semantic relationships between terms

### 8.1.5 Advantages and Limitations of Extractive Summarization

**Advantages:**
- **Simplicity** -- Straightforward to implement using basic statistical techniques
- **Efficiency** -- Computationally inexpensive; suitable for real-time applications
- **Preserves Original Text** -- Minimizes risk of introducing errors or misinterpretations

**Limitations:**
- **Coherence** -- Selected sentences may not flow smoothly together
- **Redundancy** -- Similar sentences may all be selected, leading to repetition
- **Limited Abstraction** -- Cannot generate new sentences or paraphrase content; relies entirely on what exists in the source text

---
## 8.2 Abstractive Summarization

Abstractive summarization generates **new sentences** that convey the meaning of the original text. Rather than selecting existing sentences, it rephrases and condenses the content, much like how humans naturally summarize.

**Key components:**
1. **Encoder** -- Processes the input text and converts it into a fixed-size context vector that captures the essential meaning
2. **Decoder** -- Uses the context vector to generate new summary sentences

Common architectures include RNNs, LSTMs, and Transformer-based models (BERT, GPT, BART, T5).

### 8.2.1 Abstractive Summarization with BART

**BART** (Bidirectional and Auto-Regressive Transformers) combines the strengths of bidirectional and autoregressive models. The `facebook/bart-large-cnn` model is fine-tuned on the CNN/DailyMail summarization dataset.

> **Note:** The first time you run this cell, it will download the model (~1.6 GB). This may take several minutes depending on your connection.

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

# Load the pre-trained BART model and tokenizer
model_name = "facebook/bart-large-cnn"
bart_model = BartForConditionalGeneration.from_pretrained(model_name)
bart_tokenizer = BartTokenizer.from_pretrained(model_name)

# Sample text
text = """Natural language processing (NLP) is a subfield of linguistics, computer
science, and artificial intelligence
concerned with the interactions between computers and human language, in particular
how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language
processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# Tokenize and encode the text
inputs = bart_tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate the summary
summary_ids = bart_model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("BART Summary:")
print(summary)

**Key generation parameters:**

| Parameter | Description |
|---|---|
| `max_length` | Maximum number of tokens in the generated summary |
| `min_length` | Minimum number of tokens in the generated summary |
| `length_penalty` | Higher values encourage shorter summaries |
| `num_beams` | Number of beams for beam search (improves quality) |
| `early_stopping` | Stop beam search when `num_beams` sentences are finished |

### 8.2.2 Abstractive Summarization with T5

**T5** (Text-To-Text Transfer Transformer) treats every NLP task as a text-to-text problem. For summarization, we prepend `"summarize: "` to the input. The `t5-small` variant is lightweight and suitable for experimentation.

> **Note:** The first run will download the model (~242 MB for t5-small).

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the pre-trained T5 model and tokenizer
model_name = "t5-small"
t5_model = T5ForConditionalGeneration.from_pretrained(model_name)
t5_tokenizer = T5Tokenizer.from_pretrained(model_name)

# Sample text
text = """Natural language processing (NLP) is a subfield of linguistics, computer
science, and artificial intelligence
concerned with the interactions between computers and human language, in particular
how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language
processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# Tokenize and encode the text
inputs = t5_tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate the summary
summary_ids = t5_model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("T5 Summary:")
print(summary)

### 8.2.3 Advanced Abstractive Summarization Techniques

Beyond BART and T5, several advanced approaches exist:

**Transformer-based Models:**
- **BERT** -- Reads text bidirectionally for better contextual understanding
- **GPT** -- Autoregressive model that generates fluent, contextually relevant text
- **BART** -- Combines bidirectional encoding with autoregressive decoding

**Pointer-Generator Networks:**
- Merge extractive and abstractive approaches
- Can **copy** words directly from the source (preserving accuracy for technical terms)
- Can also **generate** new words for better coherence and readability

**Reinforcement Learning:**
- Optimizes summarization through reward/penalty signals
- Reward functions can prioritize coherence, readability, or information coverage
- The model iteratively improves by maximizing these rewards

### 8.2.4 Advantages and Limitations of Abstractive Summarization

**Advantages:**
- **Coherence and Readability** -- Produces summaries that flow naturally
- **Flexibility** -- Can paraphrase and condense information creatively
- **Human-Like Summaries** -- Closer to how humans naturally summarize text

**Limitations:**
- **Complexity** -- Requires significant computational resources for training and inference
- **Training Data** -- Needs large amounts of labeled data to achieve high performance
- **Potential for Errors** -- May introduce factual inaccuracies ("hallucinations") or grammatical errors

---
## 8.3 Evaluating Summarization: ROUGE Scores

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is the standard metric for evaluating text summarization. It compares the generated summary against one or more reference summaries.

**Common ROUGE variants:**
- **ROUGE-1** -- Measures overlap of unigrams (individual words)
- **ROUGE-2** -- Measures overlap of bigrams (pairs of consecutive words)
- **ROUGE-L** -- Measures the longest common subsequence (captures sentence-level structure)

Each variant reports **Precision**, **Recall**, and **F1-score**.

In [ ]:
from rouge_score import rouge_scorer

# Reference summary (human-written "gold standard")
reference = """Natural language processing is a subfield of linguistics, computer science,
and artificial intelligence focused on interactions between computers and human language.
It involves processing and analyzing natural language data, with challenges in speech
recognition, language understanding, and generation."""

# Generated summaries to evaluate
extractive_summary = """Natural language processing (NLP) is a subfield of linguistics,
computer science, and artificial intelligence concerned with the interactions between
computers and human language, in particular how to program computers to process and
analyze large amounts of natural language data."""

abstractive_summary = """Natural language processing (NLP) is a subfield of linguistics,
computer science, and artificial intelligence that focuses on the interactions between
computers and human language. It involves programming computers to process and analyze
large amounts of natural language data and often includes tasks such as speech recognition,
natural language understanding, and natural language generation."""

# Initialize the ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Evaluate extractive summary
print("=" * 60)
print("EXTRACTIVE Summary ROUGE Scores")
print("=" * 60)
scores = scorer.score(reference, extractive_summary)
for metric, score in scores.items():
    print(f"{metric}:")
    print(f"  Precision: {score.precision:.4f}")
    print(f"  Recall:    {score.recall:.4f}")
    print(f"  F1-Score:  {score.fmeasure:.4f}")

print()

# Evaluate abstractive summary
print("=" * 60)
print("ABSTRACTIVE Summary ROUGE Scores")
print("=" * 60)
scores = scorer.score(reference, abstractive_summary)
for metric, score in scores.items():
    print(f"{metric}:")
    print(f"  Precision: {score.precision:.4f}")
    print(f"  Recall:    {score.recall:.4f}")
    print(f"  F1-Score:  {score.fmeasure:.4f}")

**Interpreting ROUGE scores:**

- **Higher is better** for all metrics
- **ROUGE-1** captures basic content overlap (did we mention the same words?)
- **ROUGE-2** captures phrase-level similarity (did we use the same word pairs?)
- **ROUGE-L** captures structural similarity via longest common subsequence
- Abstractive summaries may score differently than extractive ones -- lower ROUGE does not always mean worse quality, since abstractive methods paraphrase rather than copy

---
## 8.4 Comparison: Extractive vs. Abstractive Summarization

| Feature | Extractive | Abstractive |
|---|---|---|
| **Approach** | Selects existing sentences | Generates new sentences |
| **Coherence** | May lack flow between sentences | More natural and coherent |
| **Accuracy** | High fidelity to source | Risk of hallucinations |
| **Complexity** | Simple, fast | Complex, resource-intensive |
| **Training Data** | Minimal or none needed | Large labeled datasets required |
| **Use Cases** | News aggregation, quick overviews | Report generation, content creation |
| **Human-Likeness** | Less natural | More natural |

---
## 8.5 Practical Exercises

### Exercise 1: Extractive Summarization with NLTK

**Task:** Perform extractive summarization on the following text using term frequency.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist

# Exercise text
text = """Machine learning is a subset of artificial intelligence. It involves
algorithms and statistical models to perform tasks without explicit instructions.
Machine learning is widely used in various applications such as image recognition,
natural language processing, and autonomous driving. It relies on patterns and
inference instead of predefined rules."""

# Preprocess the text
sentences = sent_tokenize(text)
stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence):
    words = word_tokenize(sentence.lower())
    words = [word for word in words if word.isalnum() and word not in stop_words]
    return words

# Sentence scoring based on term frequency
def score_sentences(sentences):
    sentence_scores = []
    word_frequencies = FreqDist(
        [word for sentence in sentences for word in preprocess_sentence(sentence)]
    )
    for sentence in sentences:
        words = preprocess_sentence(sentence)
        sentence_score = sum(word_frequencies[word] for word in words)
        sentence_scores.append((sentence, sentence_score))
    return sentence_scores

# Select top-ranked sentences
def select_sentences(sentence_scores, num_sentences=2):
    sentence_scores.sort(key=lambda x: x[1], reverse=True)
    selected_sentences = [sentence[0] for sentence in sentence_scores[:num_sentences]]
    return selected_sentences

# Generate summary
sentence_scores = score_sentences(sentences)
summary_sentences = select_sentences(sentence_scores)
summary = ' '.join(summary_sentences)

print("Summary:")
print(summary)

### Exercise 2: Extractive Summarization with TextRank

**Task:** Perform extractive summarization on the same text using the TextRank algorithm.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.cluster.util import cosine_distance
import numpy as np
import networkx as nx

# Exercise text
text = """Machine learning is a subset of artificial intelligence. It involves
algorithms and statistical models to perform tasks without explicit instructions.
Machine learning is widely used in various applications such as image recognition,
natural language processing, and autonomous driving. It relies on patterns and
inference instead of predefined rules."""

# Preprocess the text
sentences = sent_tokenize(text)
stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence):
    words = word_tokenize(sentence.lower())
    words = [word for word in words if word.isalnum() and word not in stop_words]
    return words

# Build sentence similarity matrix
def build_similarity_matrix(sentences):
    similarity_matrix = np.zeros((len(sentences), len(sentences)))
    for i, sentence1 in enumerate(sentences):
        for j, sentence2 in enumerate(sentences):
            if i != j:
                words1 = preprocess_sentence(sentence1)
                words2 = preprocess_sentence(sentence2)
                similarity_matrix[i][j] = 1 - cosine_distance(words1, words2)
    return similarity_matrix

# Apply TextRank algorithm
def textrank(sentences, num_sentences=2):
    similarity_matrix = build_similarity_matrix(sentences)
    similarity_graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(similarity_graph)
    ranked_sentences = sorted(
        ((scores[i], sentence) for i, sentence in enumerate(sentences)),
        reverse=True
    )
    selected_sentences = [sentence for score, sentence in ranked_sentences[:num_sentences]]
    return selected_sentences

# Generate summary
summary_sentences = textrank(sentences)
summary = ' '.join(summary_sentences)

print("Summary:")
print(summary)

### Exercise 3: Abstractive Summarization with BART

**Task:** Perform abstractive summarization on the same text using the BART model.

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

# Load the pre-trained BART model and tokenizer
model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name)
tokenizer = BartTokenizer.from_pretrained(model_name)

# Exercise text
text = """Machine learning is a subset of artificial intelligence. It involves
algorithms and statistical models to perform tasks without explicit instructions.
Machine learning is widely used in various applications such as image recognition,
natural language processing, and autonomous driving. It relies on patterns and
inference instead of predefined rules."""

# Tokenize and encode the text
inputs = tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate the summary
summary_ids = model.generate(
    inputs,
    max_length=100,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Summary:")
print(summary)

### Exercise 4: Abstractive Summarization with T5

**Task:** Perform abstractive summarization on the same text using the T5 model.

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the pre-trained T5 model and tokenizer
model_name = "t5-small"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Exercise text
text = """Machine learning is a subset of artificial intelligence. It involves
algorithms and statistical models to perform tasks without explicit instructions.
Machine learning is widely used in various applications such as image recognition,
natural language processing, and autonomous driving. It relies on patterns and
inference instead of predefined rules."""

# Tokenize and encode the text
inputs = tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate the summary
summary_ids = model.generate(
    inputs,
    max_length=100,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Summary:")
print(summary)

### Exercise 5: Comparing BART and T5 Summaries

**Task:** Compare the summaries generated by BART and T5 from Exercises 3 and 4. Use ROUGE scores to evaluate which model produces a summary closer to a reference.

**Discussion points:**
- Which summary is more concise?
- Which captures the key points more effectively?
- Do the ROUGE scores align with your qualitative assessment?

In [ ]:
from rouge_score import rouge_scorer

# Reference summary
reference = """Machine learning, a subset of artificial intelligence, uses algorithms
and statistical models to perform tasks without explicit instructions. It is widely
used in image recognition, natural language processing, and autonomous driving,
relying on patterns and inference instead of predefined rules."""

# Example BART summary (replace with your actual output from Exercise 3)
bart_summary = """Machine learning, a subset of artificial intelligence, uses algorithms
and statistical models to perform tasks without explicit instructions. It is widely used
in image recognition, natural language processing, and autonomous driving, relying on
patterns and inference instead of predefined rules."""

# Example T5 summary (replace with your actual output from Exercise 4)
t5_summary = """Machine learning is a subset of artificial intelligence that uses
algorithms and statistical models to perform tasks without explicit instructions. It is
widely used in image recognition, natural language processing, and autonomous driving,
relying on patterns and inference."""

# Evaluate both summaries
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

print("BART ROUGE Scores:")
print("-" * 40)
bart_scores = scorer.score(reference, bart_summary)
for metric, score in bart_scores.items():
    print(f"  {metric}: P={score.precision:.4f} R={score.recall:.4f} F1={score.fmeasure:.4f}")

print("\nT5 ROUGE Scores:")
print("-" * 40)
t5_scores = scorer.score(reference, t5_summary)
for metric, score in t5_scores.items():
    print(f"  {metric}: P={score.precision:.4f} R={score.recall:.4f} F1={score.fmeasure:.4f}")

print("\n" + "=" * 60)
print("Discussion:")
print("Both BART and T5 produce coherent summaries. BART tends to")
print("include slightly more detail, while T5 is more concise.")
print("Replace the example summaries above with your actual outputs")
print("from Exercises 3 and 4 to see real comparisons.")

---
## Chapter Summary

In this chapter, we covered the two main approaches to text summarization:

**Extractive Summarization:**
- Selects key sentences directly from the source text
- Implemented using term frequency scoring (NLTK) and TextRank (NetworkX)
- Also explored LSA as an advanced technique using SVD
- Simple and efficient, but may lack coherence and produce redundant summaries

**Abstractive Summarization:**
- Generates new sentences that convey the meaning of the original text
- Implemented using BART and T5 models from Hugging Face Transformers
- Produces more natural, human-like summaries
- More complex and computationally intensive; risk of factual errors

**Evaluation:**
- ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L) measure overlap between generated and reference summaries
- Higher scores indicate better alignment with reference summaries
- Quantitative metrics should be supplemented with qualitative human evaluation

Both approaches have their place depending on the application requirements -- extractive for speed and accuracy, abstractive for coherence and readability.